> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.

# Chapter 4 — Hybrid Retrieval & Reranking (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2004.%20Hallucinations%20and%20RAG%20Systems/LC4LSH_Chapter_4_Hybrid_Retrieval_and_Reranking.ipynb)

**Learning objectives**
- Combine lexical (BM25) and vector retrieval with fusion
- Apply metadata filters and exact-ID lookup
- Fuse ranked lists with Reciprocal Rank Fusion (RRF)
- Rerank candidates for precision

> Runtime: ~3 min (CPU)  
> Cost: $0 (local embeddings)  
> Data: synthetic biomedical doc set


No single retriever is best for scientific search. **Vector search** captures semantics; **lexical search** nails exact IDs; **metadata filters** enforce hard constraints.

We build a hybrid pipeline: lexical + vector -> RRF -> rerank, and show when each stage wins.


## API keys & credentials

Mostly local (no paid API needed); the bootstrap sets up an optional provider for gated LLM cells.


In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
if not IN_COLAB:
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)

API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret("LC4LSH_ANTHROPIC_API_KEY", "sk-ant-...")
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")
print("API keys loaded for", API_KEY_PROVIDER)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""


## Installation (pinned)


In [ ]:
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "sentence-transformers>=2.7" "rank-bm25>=0.2.2" "numpy>=1.26,<3" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)


In [ ]:
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "lsv2_pt_...")
LANGSMITH_PROJECT = "lc4lsh-chapter4-hybrid"
REGION = "US"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith("lsv2_"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF")


## A small biomedical corpus with IDs and metadata


In [ ]:
DOCS = [
    {"id": "D1", "text": "Metformin is a first-line biguanide for type 2 diabetes.", "year": 2021, "type": "review"},
    {"id": "D2", "text": "Trial NCT04280705 evaluated metformin vs placebo in prediabetes.", "year": 2023, "type": "trial"},
    {"id": "D3", "text": "Aspirin (CC(=O)Oc1ccccc1C(=O)O) irreversibly inhibits COX-1.", "year": 2019, "type": "review"},
    {"id": "D4", "text": "Antihypertensive agents lower blood pressure via multiple mechanisms.", "year": 2022, "type": "review"},
    {"id": "D5", "text": "GLP-1 receptor agonists improve glycemic control and reduce weight.", "year": 2024, "type": "trial"},
    {"id": "D6", "text": "The SMILES for aspirin is CC(=O)Oc1ccccc1C(=O)O.", "year": 2020, "type": "reference"},
]
texts = [d["text"] for d in DOCS]
print(len(DOCS), "documents")


## 1. Lexical retrieval (BM25)


In [ ]:
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi([t.lower().split() for t in texts])
def lexical_rank(q):
    s = bm25.get_scores(q.lower().split())
    return sorted(range(len(texts)), key=lambda i: -s[i])
print("BM25:", [DOCS[i]["id"] for i in lexical_rank("aspirin SMILES CC(=O)Oc1ccccc1C(=O)O")])


## 2. Vector retrieval (local embeddings)


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")
emb = embedder.encode(texts, normalize_embeddings=True)
def vector_rank(q):
    s = emb @ embedder.encode([q], normalize_embeddings=True)[0]
    return sorted(range(len(texts)), key=lambda i: -s[i])
print("Vector:", [DOCS[i]["id"] for i in vector_rank("blood pressure medication")])


## 3. Fusion with Reciprocal Rank Fusion (RRF)


In [ ]:
def rrf(rank_lists, k=60):
    sc = {}
    for ranks in rank_lists:
        for pos, idx in enumerate(ranks):
            sc[idx] = sc.get(idx, 0.0) + 1.0 / (k + pos + 1)
    return sorted(sc, key=lambda i: -sc[i])
def hybrid_rank(q):
    return rrf([lexical_rank(q), vector_rank(q)])
for q in ["aspirin SMILES CC(=O)Oc1ccccc1C(=O)O", "blood pressure medication"]:
    print(q, "->", [DOCS[i]["id"] for i in hybrid_rank(q)])


## 4. Metadata filters (hard constraints)


In [ ]:
def hybrid_with_filter(q, **filters):
    allowed = {i for i, d in enumerate(DOCS) if all(d.get(k) == v for k, v in filters.items())}
    lo = [i for i in lexical_rank(q) if i in allowed]
    vo = [i for i in vector_rank(q) if i in allowed]
    return rrf([lo, vo])
print("type=trial:", [DOCS[i]["id"] for i in hybrid_with_filter("glycemic control", type="trial")])


## 5. Reranking (cross-encoder)


In [ ]:
try:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    def rerank(q, cands, top=3):
        scores = reranker.predict([(q, texts[i]) for i in cands])
        return [cands[i] for i in sorted(range(len(cands)), key=lambda j: -scores[j])][:top]
    q = "blood pressure medication"
    print("before:", [DOCS[i]["id"] for i in hybrid_rank(q)])
    print("after :", [DOCS[i]["id"] for i in rerank(q, hybrid_rank(q))])
except Exception as e:
    print("cross-encoder unavailable:", e)


## Limitations & safety notes

- **Fusion is not magic.** RRF assumes each retriever is informative; a noisy retriever pollutes the fusion.
- **Filters can over-constrain.** A wrong filter silently drops the correct document.
- **Cross-encoders are slow.** Rerank only top-k, never the whole corpus.


In [ ]:
# Cleanup
import gc, torch
for _v in ("embedder", "reranker", "bm25", "emb"):
    globals().pop(_v, None)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Why does RRF not need score calibration?</summary>It uses only rank positions, so heterogeneous scores reduce to comparable ranks.</details>

<details><summary>When does lexical beat vector in science?</summary>Exact identifiers - accession numbers, trial IDs, SMILES, gene symbols.</details>

<details><summary>Why rerank only top-k?</summary>Cross-encoders score each pair jointly (expensive); top-k gives most of the precision at a fraction of the cost.</details>

### Tasks
- **Task A** - Add weighted fusion (0.7 vector + 0.3 lexical) and compare to RRF on 3 queries.
- **Task B** - Implement exact-ID lookup short-circuiting on NCT IDs / SMILES.
- **Task C** - Build 5 labeled queries; report recall@3 for lexical vs vector vs hybrid.
- **Task D** - Add a year-range filter and show its effect.
